# Week 3 Production LLM Serving Benchmark: vLLM & SGLang on Colab GPU

**Repository:** [vincent-314/enterprise-ops-copilot](https://github.com/vincent-314/enterprise-ops-copilot)  
**Goal:** Quantify real-world GPU serving economics and Agent capability stability across:
1. **Concurrency Scaling (`c = 1, 4, 8, 16`)**: `TTFT P50/P95`, `TPOT P50/P95`, system decode throughput (`tok/s`), and KV-Cache metrics.
2. **Prefix Caching Speedup (`prefix_cache_rag`)**: Cold vs. Warm TTFT on a ~1,800-token shared Enterprise SRE Runbook prefix (`vLLM APC` / optional `SGLang RadixAttention`).
3. **Capability Regression (`probe_models.py`)**: Structured JSON, Tool-Call Precision, High-Risk Restraint, and Error Faithfulness under vLLM serving.


In [ ]:
#@title 1. Clone Repo, Detect Colab GPU, and Choose Optimal Precision
import os
import pathlib
import subprocess

import torch

REPO_URL = "https://github.com/vincent-314/enterprise-ops-copilot.git"
WORKDIR = pathlib.Path("/content/enterprise-ops-copilot")
if not WORKDIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(WORKDIR)], check=True)
else:
    subprocess.run(["git", "-C", str(WORKDIR), "pull", "--ff-only"], check=True)
os.chdir(WORKDIR)

assert torch.cuda.is_available(), "No GPU detected! Switch Colab Runtime -> GPU (T4/L4/A100)."
props = torch.cuda.get_device_properties(0)
GPU_NAME = props.name
VRAM_GB = round(props.total_memory / (1024**3), 1)
CC = (props.major, props.minor)

# Auto-pick optimal Qwen3-8B checkpoint & dtype for the allocated GPU
if VRAM_GB < 20 or CC < (8, 0):
    # T4 (16GB Turing, lacks native BF16/FP8 tensor cores)
    VLLM_MODEL = "Qwen/Qwen3-8B-AWQ"
    VLLM_DTYPE = "half"
    VLLM_QUANT = "awq"
    MAX_LEN = "8192"
    CONC = "1,4,8,16"
else:
    # L4 (24GB Ada) or A100 (40GB/80GB Ampere)
    VLLM_MODEL = "Qwen/Qwen3-8B-AWQ" if "L4" in GPU_NAME else "Qwen/Qwen3-8B"
    VLLM_DTYPE = "auto"
    VLLM_QUANT = "awq" if VLLM_MODEL.endswith("-AWQ") else ""
    MAX_LEN = "16384"
    CONC = "1,4,8,16"

os.environ["GPU_NAME"] = GPU_NAME
os.environ["VLLM_MODEL"] = VLLM_MODEL
os.environ["VLLM_DTYPE"] = VLLM_DTYPE
os.environ["VLLM_QUANT"] = VLLM_QUANT
os.environ["MAX_LEN"] = MAX_LEN
os.environ["CONC"] = CONC
print(f"Detected GPU: {GPU_NAME} ({VRAM_GB} GB, cc={CC})")
print(
    f"Selected: model={VLLM_MODEL}, dtype={VLLM_DTYPE}, "
    f"quant={VLLM_QUANT or 'none'}, max_len={MAX_LEN}"
)


In [ ]:
#@title 2. Install vLLM & Copilot Dependencies (~60-90s)
!pip install -q --upgrade pip
!pip install -q -e . vllm matplotlib


In [ ]:
#@title 3. Phase A - Launch vLLM Server + Concurrency Sweep + Capability Probe
import os
import pathlib
import subprocess
import time
import urllib.request

pathlib.Path("reports/colab").mkdir(parents=True, exist_ok=True)
VLLM_MODEL = os.environ["VLLM_MODEL"]
GPU_NAME = os.environ["GPU_NAME"]
CONC = os.environ["CONC"]

cmd = [
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", VLLM_MODEL,
    "--host", "127.0.0.1",
    "--port", "8000",
    "--dtype", os.environ["VLLM_DTYPE"],
    "--max-model-len", os.environ["MAX_LEN"],
    "--gpu-memory-utilization", "0.90",
    "--enable-prefix-caching",
    "--enable-auto-tool-choice",
    "--tool-call-parser", "hermes",
]
if os.environ["VLLM_QUANT"]:
    cmd.extend(["--quantization", os.environ["VLLM_QUANT"]])

log_f = open("/tmp/vllm.log", "w")
proc = subprocess.Popen(cmd, stdout=log_f, stderr=subprocess.STDOUT)


def wait_healthy(url, target_proc, log_path, timeout=480):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if target_proc.poll() is not None:
            print(pathlib.Path(log_path).read_text()[-4000:])
            raise RuntimeError(f"Server exited early with code {target_proc.returncode}")
        try:
            with urllib.request.urlopen(url, timeout=2) as r:
                if r.status == 200:
                    return True
        except Exception:
            time.sleep(3)
    print(pathlib.Path(log_path).read_text()[-4000:])
    raise TimeoutError("Server failed to become healthy in time")


try:
    wait_healthy("http://127.0.0.1:8000/health", proc, "/tmp/vllm.log")
    print("vLLM is healthy! Running concurrency & prefix-cache benchmarks...")

    subprocess.run(
        [
            "python", "scripts/bench_serving.py",
            "--base-url", "http://127.0.0.1:8000/v1",
            "--api-key", "vllm-local",
            "--model", VLLM_MODEL,
            "--engine", "vllm",
            "--gpu-name", GPU_NAME,
            "--concurrency", CONC,
            "--requests-per-stage", "16",
            "--metrics-url", "http://127.0.0.1:8000/metrics",
            "--extra-body-json", '{"chat_template_kwargs":{"enable_thinking":false}}',
            "--output-json", "reports/colab/vllm_bench.json",
            "--output-md", "reports/colab/vllm_bench.md",
        ],
        check=True,
    )

    env = os.environ.copy()
    env.update(
        {
            "PYTHONPATH": "src",
            "APP_LOCALE": "en",
            "LLM_BASE_URL": "http://127.0.0.1:8000/v1",
            "LLM_API_KEY": "vllm-local",
            "LLM_MODEL": VLLM_MODEL,
        }
    )
    probe_out = subprocess.run(
        ["python", "scripts/probe_models.py", VLLM_MODEL],
        env=env,
        capture_output=True,
        text=True,
        check=True,
    )
    print(probe_out.stdout)
    pathlib.Path("reports/colab/vllm_probe.txt").write_text(probe_out.stdout, encoding="utf-8")
finally:
    proc.terminate()
    try:
        proc.wait(timeout=20)
    except subprocess.TimeoutExpired:
        proc.kill()
    log_f.close()
    print("vLLM server stopped cleanly.")


In [ ]:
#@title 4. (Optional) Phase B - Benchmark SGLang RadixAttention on the Same GPU
#@markdown Set `RUN_SGLANG = True` if you also want an apples-to-apples SGLang RadixAttention run.
RUN_SGLANG = False  #@param {type:"boolean"}

if RUN_SGLANG:
    subprocess.run(["pip", "install", "-q", "sglang[all]"], check=True)
    VLLM_MODEL = os.environ["VLLM_MODEL"]
    GPU_NAME = os.environ["GPU_NAME"]
    CONC = os.environ["CONC"]

    sg_cmd = [
        "python", "-m", "sglang.launch_server",
        "--model-path", VLLM_MODEL,
        "--host", "127.0.0.1",
        "--port", "8001",
        "--dtype", "float16" if os.environ["VLLM_DTYPE"] == "half" else "auto",
        "--context-length", os.environ["MAX_LEN"],
        "--mem-fraction-static", "0.88",
        "--tool-call-parser", "qwen25",
        "--enable-metrics",
    ]
    if os.environ["VLLM_QUANT"]:
        sg_cmd.extend(["--quantization", os.environ["VLLM_QUANT"]])

    sg_log = open("/tmp/sglang.log", "w")
    sg_proc = subprocess.Popen(sg_cmd, stdout=sg_log, stderr=subprocess.STDOUT)
    try:
        t0 = time.time()
        while time.time() - t0 < 480:
            if sg_proc.poll() is not None:
                print(pathlib.Path("/tmp/sglang.log").read_text()[-4000:])
                raise RuntimeError("SGLang exited early")
            try:
                with urllib.request.urlopen("http://127.0.0.1:8001/health", timeout=2) as r:
                    if r.status == 200:
                        break
            except Exception:
                time.sleep(3)
        subprocess.run(
            [
                "python", "scripts/bench_serving.py",
                "--base-url", "http://127.0.0.1:8001/v1",
                "--api-key", "sglang-local",
                "--model", VLLM_MODEL,
                "--engine", "sglang",
                "--gpu-name", GPU_NAME,
                "--concurrency", CONC,
                "--requests-per-stage", "16",
                "--metrics-url", "http://127.0.0.1:8001/metrics",
                "--extra-body-json", '{"chat_template_kwargs":{"enable_thinking":false}}',
                "--output-json", "reports/colab/sglang_bench.json",
                "--output-md", "reports/colab/sglang_bench.md",
            ],
            check=True,
        )
    finally:
        sg_proc.terminate()
        try:
            sg_proc.wait(timeout=20)
        except subprocess.TimeoutExpired:
            sg_proc.kill()
        sg_log.close()
        print("SGLang server stopped cleanly.")


In [ ]:
#@title 5. Plot Comparison Charts & Download Result Bundle
import json
import pathlib
import zipfile

import matplotlib.pyplot as plt

results = [
    json.loads(p.read_text())
    for p in sorted(pathlib.Path("reports/colab").glob("*_bench.json"))
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for r in results:
    eng = f"{r['metadata']['engine']} ({r['metadata']['gpu_name']})"
    triage = [s for s in r["summaries"] if s["workload"] == "short_triage"]
    concs = [s["concurrency"] for s in triage]
    axes[0].plot(concs, [s["ttft_p95_ms"] for s in triage], marker="o", label=eng)
    axes[1].plot(concs, [s["system_throughput_tok_s"] for s in triage], marker="o", label=eng)

axes[0].set_title("TTFT P95 vs Concurrency (short_triage)")
axes[0].set_xlabel("Concurrency")
axes[0].set_ylabel("ms")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title("System Throughput vs Concurrency")
axes[1].set_xlabel("Concurrency")
axes[1].set_ylabel("tok/s")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

labels = []
cold_vals = []
warm_vals = []
for r in results:
    eng = r["metadata"]["engine"]
    cold = next((s["ttft_p50_ms"] for s in r["summaries"] if s["workload"] == "prefix_cache_rag_cold"), 0)
    warm = next((s["ttft_p50_ms"] for s in r["summaries"] if s["workload"] == "prefix_cache_rag_warm"), 0)
    labels.append(eng)
    cold_vals.append(cold)
    warm_vals.append(warm)
x = range(len(labels))
axes[2].bar([i - 0.18 for i in x], cold_vals, width=0.35, label="Cold Prefix")
axes[2].bar([i + 0.18 for i in x], warm_vals, width=0.35, label="Warm Prefix")
axes[2].set_xticks(list(x))
axes[2].set_xticklabels(labels)
axes[2].set_title("Prefix Cache TTFT P50 (Cold vs Warm)")
axes[2].set_ylabel("ms")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("reports/colab/week3_benchmark_charts.png", dpi=150)
plt.show()

bundle_path = pathlib.Path("reports/week3_colab_results.zip")
with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in pathlib.Path("reports/colab").iterdir():
        zf.write(f, arcname=f.name)
print(f"Created {bundle_path}")
try:
    from google.colab import files
    files.download(str(bundle_path))
except ImportError:
    print("Not running inside Google Colab UI; bundle saved locally at", bundle_path)
